# Домашнее задание №2

**Домашнее задание 1**

Посчитать количество рейтингов больше 4 для каждого фильма и вывести фильмы в порядке убывания количества этих оценок (можно вывести топ 10)

Файл можете взять из прошлого домашнего задания + сохраните результат на диск

In [1]:
import heapq
import pickle

from pyspark.sql import SparkSession

In [2]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("HomeWork2")
    .config("spark.ui.port", "4050")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    # .config("spark.dynamicAllocation.enabled", "true")
    # .config("spark.shuffle.service.enabled", "true")  # трекер, чтобы возвращать ресурсы
    .getOrCreate()
)
sc = spark.sparkContext

In [3]:
rdd = sc.textFile("user_ratedmovies.dat")
header = rdd.first()
data = rdd.map(lambda line: line.strip()).filter(lambda line: line != "" and line != header)

In [4]:
movie_stats = (
    data
    .map(lambda line: line.split("\t")[:3])
    .map(lambda p: (p[1], float(p[2])))
    .aggregateByKey(
        (0.0, 0),
        lambda acc, r: (acc[0] + r, acc[1] + 1),
        lambda a, b: (a[0] + b[0], a[1] + b[1])
    )
    .mapValues(lambda sum_count: (sum_count[0] / sum_count[1], sum_count[1]))
)

In [5]:
result = (
    movie_stats
    .filter(lambda x: x[1][0] > 4)
    .takeOrdered(20, key=lambda x: -x[1][1])
)

for movie_id, (avg, cnt) in result:
    print(f"movie_id={movie_id}, avg_rating={avg:.2f}, count={cnt}")

movie_id=2571, avg_rating=4.17, count=1670
movie_id=4993, avg_rating=4.08, count=1576
movie_id=296, avg_rating=4.24, count=1537
movie_id=5952, avg_rating=4.03, count=1528
movie_id=2858, avg_rating=4.11, count=1472
movie_id=7153, avg_rating=4.09, count=1457
movie_id=318, avg_rating=4.37, count=1441
movie_id=2959, avg_rating=4.25, count=1434
movie_id=593, avg_rating=4.05, count=1431
movie_id=260, avg_rating=4.08, count=1365
movie_id=4226, avg_rating=4.18, count=1320
movie_id=1196, avg_rating=4.10, count=1306
movie_id=1198, avg_rating=4.14, count=1304
movie_id=50, avg_rating=4.29, count=1218
movie_id=47, avg_rating=4.06, count=1206
movie_id=7361, avg_rating=4.10, count=1202
movie_id=527, avg_rating=4.17, count=1170
movie_id=32, avg_rating=4.01, count=1158
movie_id=1136, avg_rating=4.13, count=1151
movie_id=4973, avg_rating=4.19, count=1137


Домашнее задание 2

В этом задании у вас есть файл с обученным word2vec на произведении Достоевского Преступление и наказание. Файл - list, каждый элемент которого слово и его вектор в формате `('word', [vector])`. Необходимо для каждого слова собрать список его top-10 похожих слов по косинусной метрике. Результат также сохраните на диск и выведите синонимы для слова 'топор' и 'деньга'. Файл в `pickle`, так что для считывания воспользуйтесь не spark, можете взять любимый `pandas`.

Файл не очень уж и маленький, рекомендую сначала написать код на кошках/собачках, а потом уже на всём, так как ядра 2, считаться будет очень долго. В качестве одного из вариантов можете рассмотреть метод `cartesian()`.

In [6]:
with open("w2v_vectors.pickle", "rb") as f:
    data = pickle.load(f)

In [7]:
rdd = sc.parallelize(data, numSlices=8)


def normalize(item):
    word, vector = item
    amount = sum(x * x for x in vector) ** 0.5
    return (word, [x / amount for x in vector] if amount > 0 else vector)


normalized = rdd.map(normalize)
collected = normalized.collect()
broadcasted = sc.broadcast(collected)


def top10(item):
    word1, vector1 = item
    best = heapq.nlargest(
        10,
        (
            (sum(x1 * x2 for x1, x2 in zip(vector1, vector2)), word2)
            for word2, vector2 in broadcasted.value
            if word2 != word1
        ),
    )
    return (word1, [word2 for _, word2 in best])


result = normalized.map(top10)


final = result.collect()
for word, top in final:
    if word in ("топор", "деньга"):
        print(word, "->", top)

деньга -> ['вещь', 'отдать', 'убить', 'значит', 'известно', 'отец', 'получить', 'ограбить', 'копейка', 'полечка']
топор -> ['левый', 'старуха', 'колено', 'побежать', 'ломать', 'бросить', 'схватить', 'обнять', 'бить', 'положить']
